In [ ]:
# --- Importaciones ---------------------------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer, load_wine
from sklearn.model_selection import (
    train_test_split, cross_val_score, StratifiedKFold
)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier,
    VotingClassifier, BaggingClassifier, StackingClassifier,
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report

import sklearn
print(f'numpy:   {np.__version__}')
print(f'sklearn: {sklearn.__version__}')


In [ ]:
# --- Breast Cancer: clasificación binaria ---------------------------
bc = load_breast_cancer()
X_bc = bc.data
y_bc = bc.target

semilla = 42
X_train, X_test, y_train, y_test = train_test_split(
    X_bc, y_bc,
    test_size=0.2, random_state=semilla, stratify=y_bc
)

# Escalar (necesario para LR, SVM y KNN en el ensemble)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Train: {X_train_sc.shape} | Test: {X_test_sc.shape}')


In [ ]:
# --- Benchmark: cada modelo por separado ----------------------------
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=semilla)

modelos_base = {
    'LogisticReg':    LogisticRegression(max_iter=1000,
                                        random_state=semilla),
    'RandomForest':   RandomForestClassifier(n_estimators=100,
                                            random_state=semilla,
                                            n_jobs=-1),
    'KNN(k=5)':       KNeighborsClassifier(n_neighbors=5,
                                            n_jobs=-1),
    'SVC(rbf)':       SVC(kernel='rbf', probability=True,
                        random_state=semilla),
    'GradientBoosting': GradientBoostingClassifier(
        n_estimators=100, random_state=semilla
    ),
}

print(f'{'Modelo':<20} {'CV Acc':>8} {'±':>6}')
resultados_base = {}
for nombre, modelo in modelos_base.items():
    scores = cross_val_score(
        modelo, X_train_sc, y_train, cv=cv, scoring='accuracy'
    )
    resultados_base[nombre] = scores.mean()
    print(f'{nombre:<20} {scores.mean():>8.4f} {scores.std():>6.4f}')


In [ ]:
# --- Hard voting: clase más votada ----------------------------------
voting_hard = VotingClassifier(
    estimators=[
        ('lr',  LogisticRegression(max_iter=1000,
                                random_state=semilla)),
        ('rf',  RandomForestClassifier(n_estimators=100,
                                        random_state=semilla,
                                        n_jobs=-1)),
        ('knn', KNeighborsClassifier(n_neighbors=5, n_jobs=-1)),
    ],
    voting='hard',
)

scores_hard = cross_val_score(
    voting_hard, X_train_sc, y_train, cv=cv, scoring='accuracy'
)
print(f'Hard voting | CV: {scores_hard.mean():.4f} ± {scores_hard.std():.4f}')


In [ ]:
# --- Soft voting: promedio de probabilidades ------------------------
# SVC requiere probability=True para devolver predict_proba()
voting_soft = VotingClassifier(
    estimators=[
        ('lr',  LogisticRegression(max_iter=1000,
                                random_state=semilla)),
        ('rf',  RandomForestClassifier(n_estimators=100,
                                        random_state=semilla,
                                        n_jobs=-1)),
        ('svc', SVC(kernel='rbf', probability=True,
                    random_state=semilla)),
    ],
    voting='soft',
)

scores_soft = cross_val_score(
    voting_soft, X_train_sc, y_train, cv=cv, scoring='accuracy'
)
print(f'Soft voting | CV: {scores_soft.mean():.4f} ± {scores_soft.std():.4f}')


In [ ]:
# --- BaggingClassifier: bootstrap + promedio -----------------------
bagging = BaggingClassifier(
    estimator=DecisionTreeClassifier(random_state=semilla),
    n_estimators=100,
    max_samples=0.8,     # 80% del dataset por árbol
    max_features=0.8,    # 80% de las features por árbol
    random_state=semilla,
    n_jobs=-1,
)

scores_bag = cross_val_score(
    bagging, X_train_sc, y_train, cv=cv, scoring='accuracy'
)
print(f'Bagging DT | CV: {scores_bag.mean():.4f} ± {scores_bag.std():.4f}')


In [ ]:
# --- StackingClassifier: dos capas ----------------------------------
modelos_capa1 = [
    ('lr',  LogisticRegression(max_iter=1000,
                            random_state=semilla)),
    ('rf',  RandomForestClassifier(n_estimators=100,
                                    random_state=semilla,
                                    n_jobs=-1)),
    ('knn', KNeighborsClassifier(n_neighbors=5, n_jobs=-1)),
    ('svc', SVC(kernel='rbf', probability=True,
                random_state=semilla)),
]

# Meta-modelo: aprende a combinar las predicciones de la capa 1
meta_modelo = LogisticRegression(max_iter=1000, random_state=semilla)

stacking = StackingClassifier(
    estimators=modelos_capa1,
    final_estimator=meta_modelo,
    cv=5,                    # cross-val interna para evitar leakage
    passthrough=False,       # solo usa preds de capa 1, no X original
    n_jobs=-1,
)

scores_stack = cross_val_score(
    stacking, X_train_sc, y_train, cv=cv, scoring='accuracy'
)
print(f'Stacking   | CV: {scores_stack.mean():.4f} ± {scores_stack.std():.4f}')


In [ ]:
# --- Inspeccionar el meta-modelo ------------------------------------
# Entrenar el stacking sobre todo X_train para inspeccionar
stacking.fit(X_train_sc, y_train)

# El meta-modelo recibió 4 features (una por modelo base)
# Sus coeficientes indican en qué modelo confía más
coefs = stacking.final_estimator_.coef_[0]
nombres_capa1 = [nombre for nombre, _ in modelos_capa1]

for nombre, coef in zip(nombres_capa1, coefs):
    print(f'{nombre:<8}: {coef:+.4f}')

# Evaluación final en test
acc_stack = accuracy_score(y_test, stacking.predict(X_test_sc))
print(f'Accuracy en test: {acc_stack:.4f}')


In [ ]:
# --- Evaluar todos en test ------------------------------------------
# Entrenar los ensembles sobre X_train completo
# Definir todos los modelos en un dict para evaluar juntos
comparativa = {
    'LogisticReg (base)':   modelos_base['LogisticReg'],
    'RandomForest (base)':  modelos_base['RandomForest'],
    'Hard Voting':          voting_hard,
    'Soft Voting':          voting_soft,
    'Bagging DT':           bagging,
    'Stacking':             stacking,
}

print(f"{'Modelo':<24} {'Test Acc':>10}")
for nombre, modelo in comparativa.items():
    modelo.fit(X_train_sc, y_train)   # re-entrena sobre el train completo
    acc = accuracy_score(y_test, modelo.predict(X_test_sc))
    print(f'{nombre:<24} {acc:>10.4f}')

In [ ]:
# --- Wine: 3 clases, 13 features ------------------------------------
wine = load_wine()
X_wine = StandardScaler().fit_transform(wine.data)
y_wine = wine.target

stacking_wine = StackingClassifier(
    estimators=[
        ('lr',  LogisticRegression(max_iter=1000,
                                random_state=semilla)),
        ('rf',  RandomForestClassifier(n_estimators=100,
                                        random_state=semilla,
                                        n_jobs=-1)),
        ('gb',  GradientBoostingClassifier(n_estimators=100,
                                            random_state=semilla)),
    ],
    final_estimator=LogisticRegression(max_iter=1000,
                                        random_state=semilla),
    cv=5,
    n_jobs=-1,
)

cv_wine = StratifiedKFold(n_splits=5, shuffle=True, random_state=semilla)
scores_wine = cross_val_score(
    stacking_wine, X_wine, y_wine, cv=cv_wine, scoring='accuracy'
)
print(f'Stacking Wine | CV: {scores_wine.mean():.4f} ± {scores_wine.std():.4f}')
# Resultados típicos:
# Stacking Wine | CV: 0.9888 ± 0.0139
